# Filter and Reformat (0302)

1. Preview one transformed sample.
2. Save transformed outputs for all JSON files.


In [ ]:
from pathlib import Path
import json
import re

SRC_ROOT = Path('/home/hyeseojeon/data/Graph-RAG/results/answer/0308')
DST_ROOT = Path('/home/hyeseojeon/data/Graph-RAG/results/analysis/0308/answer')

def format_response_text(response):
    if not isinstance(response, str):
        return response

    text = response.strip()
    # Keep each quoted triplet/sentence on its own line.
    text = re.sub(r'"\s*,\s*"', '"\n"', text)
    # Also split plain prose by sentence boundaries.
    text = re.sub(r'(?<=[.!?])\s+(?=[\"A-Za-z0-9가-힣(])', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

def merge_per_document_text(graph, is_doc_graph=False):
    g = dict(graph or {})
    per_docs = g.get('per_document') or []
    docs = [
        d.get('document', '').strip()
        for d in per_docs
        if isinstance(d, dict) and d.get('document')
    ]

    retrieval = g.get('retrieval_info') or {}
    is_gold_list = retrieval.get('is_gold_list') if isinstance(retrieval, dict) else None

    g.pop('per_document', None)
    g.pop('retrieval_info', None)

    ordered = {}
    # doc_graph only: keep is_gold_list as the first key
    if is_doc_graph and is_gold_list is not None:
        ordered['is_gold_list'] = is_gold_list

    # Put documents above triples
    if docs:
        ordered['documents'] = '\n'.join(docs)

    for key in g:
        if key in {'is_gold_list', 'documents'}:
            continue
        ordered[key] = g[key]

    return ordered

def transform_item(item: dict):
    infill = item.get('infill_result') or {}
    if infill.get('ent_exist_flag') is not True:
        return None

    qg = dict(item.get('question_graph') or {})
    qg.pop('definition_triples', None)

    dg = merge_per_document_text(item.get('doc_graph') or {}, is_doc_graph=True)
    dg.pop('definition_triples', None)

    gg = merge_per_document_text(item.get('gold_graph') or {})
    gg.pop('definition_triples', None)

    reduced_infill = {}
    if 'response' in infill:
        reduced_infill['response'] = format_response_text(infill['response'])

    out = {}
    skip_top = {
        'gold_id_list',
        'predicted_answer',
        'graph_evidence',
        'answer_prompt',
        'infill_result',
        'em_score',
        'f1_score',
        'answer_skipped',
        'answer_skip_reason'
    }

    for k, v in item.items():
        if k in skip_top:
            continue

        if k == 'question_graph':
            out[k] = qg
        elif k == 'doc_graph':
            out[k] = dg
        elif k == 'gold_graph':
            out[k] = gg
        else:
            out[k] = v

        if k == 'answer_aliases':
            out['infill_result'] = reduced_infill
            if 'em_score' in item:
                out['em_score'] = item['em_score']
            if 'f1_score' in item:
                out['f1_score'] = item['f1_score']

    if 'infill_result' not in out:
        out['infill_result'] = reduced_infill
        if 'em_score' in item:
            out['em_score'] = item['em_score']
        if 'f1_score' in item:
            out['f1_score'] = item['f1_score']

    return out


In [5]:
# Step 1) Preview one sample before saving all
sample_file = sorted(SRC_ROOT.rglob('*.json'))[0]
with sample_file.open('r', encoding='utf-8') as f:
    sample_data = json.load(f)

if isinstance(sample_data, list):
    transformed_sample = next((t for t in (transform_item(x) for x in sample_data) if t is not None), None)
else:
    transformed_sample = transform_item(sample_data)

print('sample_file:', sample_file)
print('sample_found:', transformed_sample is not None)
if transformed_sample is not None:
    print('sample_keys:', list(transformed_sample.keys()))

    # JSON preview (shows escaped newlines)
    print('\n[JSON preview]')
    print(json.dumps(transformed_sample, ensure_ascii=False, indent=2)[:1500])

    # Raw response preview (shows actual line breaks)
    resp = transformed_sample.get('infill_result', {}).get('response')
    if isinstance(resp, str):
        print('\n[Raw infill_result.response]')
        print(resp[:3000])
else:
    print('No record with infill_result.ent_exist_flag == True in this file.')


sample_file: /home/hyeseojeon/data/Graph-RAG/results/baseline/0308/2wiki/Qwen2.5-14B-Instruct/baseline_2wiki_full_triplets_train_sampled_combined_all.json
sample_found: False
No record with infill_result.ent_exist_flag == True in this file.


In [3]:
# Step 2) Save all transformed files
DST_ROOT.mkdir(parents=True, exist_ok=True)
summary = []

for src in sorted(SRC_ROOT.rglob('*.json')):
    rel = src.relative_to(SRC_ROOT)
    dst = DST_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    with src.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        transformed = [y for y in (transform_item(x) for x in data) if y is not None]
        before_n = len(data)
        after_n = len(transformed)
    else:
        one = transform_item(data)
        transformed = [] if one is None else one
        before_n = 1
        after_n = 0 if transformed == [] else 1

    with dst.open('w', encoding='utf-8') as f:
        json.dump(transformed, f, ensure_ascii=False, indent=2)

    summary.append((str(rel), before_n, after_n))

print('processed files:', len(summary))
for rel, b, a in summary:
    print(f'- {rel}: {b} -> {a}')


processed files: 36
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_combined_all_all.json: 500 -> 48
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_combined_gold_all.json: 500 -> 37
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_doc_only_all_all.json: 500 -> 28
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_doc_only_gold_all.json: 500 -> 27
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_triplet_only_all_all.json: 500 -> 58
- 2wiki/Qwen2.5-14B-Instruct/answer_2wiki_triplet_only_triplet_only_gold_all.json: 500 -> 53
- 2wiki/Qwen2.5-7B-Instruct/answer_2wiki_triplet_only_combined_all_all.json: 500 -> 37
- 2wiki/Qwen2.5-7B-Instruct/answer_2wiki_triplet_only_combined_gold_all.json: 500 -> 50
- 2wiki/Qwen2.5-7B-Instruct/answer_2wiki_triplet_only_doc_only_all_all.json: 500 -> 31
- 2wiki/Qwen2.5-7B-Instruct/answer_2wiki_triplet_only_doc_only_gold_all.json: 500 -> 27
- 2wiki/Qwen2.5-7B-Instruct/answer_2wiki_triplet_only_triplet_only_all_all.json: 500 -> 27
